In [1]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [3]:
df = pd.read_csv("/Users/kummarithirumalaraju/Desktop/netflix-recommendation-engine/data/netflix_ratings.csv")

df.head()

,CustomerID,MovieID,Rating,Title,Genre,Year
0,1,365,4,R.E.M.: Tourfilm,Drama,1989.0
1,1,75,4,Grind,Drama,1997.0
2,1,379,5,Crash Dive,Drama,1996.0
3,1,157,4,Laird: White Knuckle Extreme,Drama,2004.0
4,1,106,4,Stevie Ray Vaughan and Double Trouble: Live at...,Drama,2004.0


In [4]:
df.columns

Index(['CustomerID', 'MovieID', 'Rating', 'Title', 'Genre', 'Year'], dtype='str')

In [5]:
df = df.drop_duplicates()

df["CustomerID"] = df["CustomerID"].astype(int)
df["MovieID"] = df["MovieID"].astype(int)
df["Rating"] = df["Rating"].astype(int)
df["Year"] = df["Year"].astype(int)

df.head()

,CustomerID,MovieID,Rating,Title,Genre,Year
0,1,365,4,R.E.M.: Tourfilm,Drama,1989
1,1,75,4,Grind,Drama,1997
2,1,379,5,Crash Dive,Drama,1996
3,1,157,4,Laird: White Knuckle Extreme,Drama,2004
4,1,106,4,Stevie Ray Vaughan and Double Trouble: Live at...,Drama,2004


In [6]:
movie_stats = df.groupby(["MovieID", "Title", "Genre", "Year"]).agg(
    avg_rating=("Rating", "mean"),
    rating_count=("Rating", "count")
).reset_index()

movie_stats["popularity_score"] = (
    movie_stats["avg_rating"] * np.log1p(movie_stats["rating_count"])
)

movie_stats.head()

,MovieID,Title,Genre,Year,avg_rating,rating_count,popularity_score
0,1,Dinosaur Planet,Sci-Fi,2003,3.750000,12,9.618560
1,2,Isle of Man TT 2004 Review,Drama,2004,4.000000,14,10.832201
2,3,Character,Drama,1997,3.230769,13,8.526185
3,4,Paula Abdul's Get Up & Dance,Drama,1994,3.444444,9,7.931126
4,5,The Rise and Fall of ECW,Drama,2004,3.375000,8,7.415633


In [7]:
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    df[["CustomerID", "MovieID", "Rating"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

model = SVD()
model.fit(trainset)

predictions = model.test(testset)

accuracy.rmse(predictions)
accuracy.mae(predictions)

RMSE: 1.1778
MAE:  0.9777


0.9776583143816405

In [8]:
def hybrid_recommendation(user_id, top_n=10):
    all_movies = df["MovieID"].unique()

    watched_movies = df[df["CustomerID"] == user_id]["MovieID"].unique()

    unwatched_movies = [
        movie for movie in all_movies 
        if movie not in watched_movies
    ]

    user_data = df[df["CustomerID"] == user_id]

    favorite_genres = user_data.groupby("Genre")["Rating"].mean().to_dict()

    recommendations = []

    max_popularity = movie_stats["popularity_score"].max()

    for movie_id in unwatched_movies:
        movie_row = movie_stats[movie_stats["MovieID"] == movie_id]

        if movie_row.empty:
            continue

        movie_info = movie_row.iloc[0]

        predicted_rating = model.predict(user_id, movie_id).est

        genre_score = favorite_genres.get(movie_info["Genre"], 3)

        movie_popularity = movie_info["popularity_score"]

        normalized_popularity = movie_popularity / max_popularity

        final_score = (
            0.5 * predicted_rating +
            0.3 * normalized_popularity +
            0.2 * genre_score
        )

        recommendations.append([
            movie_id,
            movie_info["Title"],
            movie_info["Genre"],
            movie_info["Year"],
            round(predicted_rating, 2),
            round(movie_popularity, 2),
            round(final_score, 2)
        ])

    result = pd.DataFrame(
        recommendations,
        columns=[
            "MovieID",
            "Title",
            "Genre",
            "Year",
            "Predicted Rating",
            "Popularity Score",
            "Hybrid Score"
        ]
    )

    return result.sort_values(
        "Hybrid Score",
        ascending=False
    ).head(top_n)

In [9]:
hybrid_recommendation(1)

,MovieID,Title,Genre,Year,Predicted Rating,Popularity Score,Hybrid Score
300,115,Lord Peter Wimsey: Murder Must Advertise,Thriller,1973,4.40,11.28,3.49
396,408,Nightbreed,Horror,1990,4.14,10.84,3.34
450,173,The Devil's Brigade,Horror,1968,4.13,10.17,3.32
210,251,Midsomer Murders: Strangler's Wood,Thriller,2000,4.04,10.96,3.30
108,461,Nightwalker #1: Midnight Detective,Horror,2000,3.98,10.96,3.27
340,80,Winter Kills,Thriller,1979,4.00,8.81,3.22
227,151,Sleepover Nightmare,Horror,2005,3.89,8.63,3.17
382,325,Ghosts of Rwanda: Frontline,Horror,2004,3.88,8.32,3.15
32,292,Saturday Night Live: The Best of Gilda Radner,Horror,2005,3.88,8.32,3.15
199,79,The Killing,Thriller,1956,3.75,10.17,3.13
